# Faster R-CNN Backbone Experiment — ResNet-50 FPN v2 vs. MobileNetV3-Large FPN

The three earlier Faster R-CNN notebooks all hold the backbone fixed at ResNet-50 FPN v2 and vary
data volume, resolution, LR schedule and sampling. This notebook varies **only the backbone** and
holds everything else constant, so the difference between the two result columns is attributable to
the feature extractor and nothing else.

**The experiment.** Two detectors, trained back to back in the same session:

| | ResNet-50 FPN v2 | MobileNetV3-Large FPN |
|---|---|---|
| Constructor | `fasterrcnn_resnet50_fpn_v2` | `fasterrcnn_mobilenet_v3_large_fpn` |
| COCO-pretrained weights | `FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT` | `FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT` |
| Backbone design | 25M-param residual CNN, FPN over 4 stages | depthwise-separable convs + squeeze-excite, FPN over 2 stages |
| Params (COCO head) | 43.7M | 19.4M |
| COCO box mAP (torchvision) | 46.7 | 32.8 |
| Role here | accuracy reference | speed/size challenger |

(Param counts are torchvision's COCO-head figures; swapping in the 148-class predictor shifts both
slightly, and the notebook prints the counts it actually built.)

**Held constant across both arms** — this is what makes the comparison a comparison:

- the same training subset, splits, taxonomy and seed;
- the same input resolution (`MIN_SIZE`/`MAX_SIZE` are passed explicitly, overriding each
  constructor's own default — see the model section, this is the easiest thing to get wrong here);
- the same batch size, LR, optimizer, cosine schedule, epoch budget and early-stopping rule;
- the same class-balanced sampler and horizontal-flip augmentation;
- the same evaluation: COCO mAP on the held-out test split, and single-stream latency at batch
  size 1 with CUDA synchronisation.

**What is deliberately *not* equalised, and why.** Learning rate is shared rather than tuned per
backbone. torchvision's own reference recipes use different LRs for these two models, so a shared
LR mildly favours whichever backbone happens to suit it. Tuning each arm separately would confound
"which backbone" with "whose LR search was luckier", and doubles the compute again; a shared,
principled LR (linear scaling rule off torchvision's `lr=0.02 @ batch 16`) is the cleaner control.
This is called out again in the closing section, since it bounds how strong a claim the result
supports.

**Compute budget.** Two full trainings in one Colab session is the binding constraint, so this run
uses the 30% stratified subset from runs 1–2 rather than the full split used by run 3. That means
the absolute mAP here is *not* comparable to run 3's full-data numbers — the two columns are
comparable to **each other**, which is the question being asked. Every artifact (checkpoints,
metrics JSON) is written per backbone under its own tag, so a disconnect part-way through never
loses the arm that already finished, and re-running the training cell resumes where it stopped.

In [ ]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

!pip install torchmetrics pycocotools pyyaml -q

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from PIL import Image
import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from torchmetrics.detection.mean_ap import MeanAveragePrecision

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("torchvision:", torchvision.__version__)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Data & Class List

Identical to the earlier Faster R-CNN notebooks: same merged YOLO-format dataset, same
`train/val/test` split lists, same 0-indexed class list shifted by +1 so label 0 stays reserved for
background. Nothing here is specific to the backbone experiment — reusing it unchanged is what
keeps this notebook's numbers on the same footing as the other runs'.

In [ ]:
MERGED_YOLO_DIR = Path("/content/drive/Shareddrives/Computer Vision Final Project/Data/merged_yolo")

TRAIN_LIST = MERGED_YOLO_DIR / "train.txt"
VAL_LIST = MERGED_YOLO_DIR / "val.txt"
TEST_LIST = MERGED_YOLO_DIR / "test.txt"

assert TRAIN_LIST.exists(), (
    f"train.txt not found at {TRAIN_LIST} - update MERGED_YOLO_DIR to match your Drive layout."
)


def load_class_names(merged_dir: Path) -> list[str]:
    classes_txt = merged_dir / "classes.txt"
    if classes_txt.exists():
        return [line.strip() for line in classes_txt.read_text().splitlines() if line.strip()]

    data_yaml = merged_dir / "data.yaml"
    if data_yaml.exists():
        import yaml
        with open(data_yaml) as f:
            names = yaml.safe_load(f)["names"]
        return list(names.values()) if isinstance(names, dict) else list(names)

    raise FileNotFoundError(
        f"Neither classes.txt nor data.yaml found in {merged_dir} - add one, or edit "
        f"load_class_names() to point at wherever the class list actually lives."
    )


CLASS_NAMES = load_class_names(MERGED_YOLO_DIR)
# Faster R-CNN reserves label 0 for background, so raw (0-indexed) label-file class ids are
# shifted by +1 when building targets; CLASS_NAMES itself stays 0-indexed.
NUM_CLASSES = len(CLASS_NAMES) + 1

print(f"{len(CLASS_NAMES)} classes (+1 background) = {NUM_CLASSES} model outputs")
print(CLASS_NAMES)

In [ ]:
import os
import shutil
import shlex
import subprocess


def image_to_label_path(image_path: Path) -> Path:
    """Swap the rightmost 'images' path segment for 'labels' and the extension for .txt."""
    parts = list(image_path.parts)
    try:
        idx = len(parts) - 1 - parts[::-1].index("images")
    except ValueError:
        raise ValueError(f"No 'images' path segment found in {image_path}")
    parts[idx] = "labels"
    return Path(*parts).with_suffix(".txt")


def load_image_list(list_path: Path) -> list[Path]:
    return [Path(line.strip()) for line in list_path.read_text().splitlines() if line.strip()]


# Staging data onto local Colab disk, via tar archives rather than a per-file copy.
#
# Reading images straight off the Drive FUSE mount makes each __getitem__ a pair of network file
# opens, so Drive's per-file latency - not GPU compute - sets the epoch time. Copying to /content
# fixes that, but a file-by-file `shutil.copytree` of ~27k images is itself slow enough to eat a
# whole Colab session. So: tar each directory into a single archive ONCE, kept on Drive and reused
# by every later session; each session then does one large sequential read (which Drive handles far
# better than thousands of small ones) plus a fast local extract.
#
# Robustness properties, all learned the hard way:
#   - Subprocess output is CAPTURED and printed. In Colab a subprocess writes to the kernel's
#     stderr, which does not appear in cell output, so a bare `check=True` surfaces only
#     "exit status 1" and throws away tar's actual explanation.
#   - GNU tar exit status 1 is NON-FATAL (2 is fatal). On a Drive FUSE mount, walking a directory
#     routinely trips "file changed as we read it" because Drive updates directory metadata as it
#     goes - the archive itself is fine. Treating exit 1 as failure discarded a complete 0.70 GB
#     archive, so exit 1 is now accepted and its warnings printed rather than swallowed.
#     (Suppressing it via --warning=no-file-changed is deliberately NOT used: that is a
#     GNU-tar-only option which hard-errors on other tar builds, and tolerating the exit status is
#     what actually matters. Showing the warnings is also more honest than hiding them.)
#   - Accepting exit 1 is only safe because a new archive is verified before the .partial is
#     renamed, so a genuinely truncated file can never be cached on Drive and trusted later.
#   - If the Drive archive cannot be built at all, staging falls back to streaming the data
#     straight from Drive to local disk with no intermediate archive - no Drive space, still fast.
DRIVE_DATA_ROOT = MERGED_YOLO_DIR.parent
LOCAL_DATA_ROOT = Path("/content/Data")
LOCAL_SCRATCH = LOCAL_DATA_ROOT.parent  # where archives land before being extracted
ARCHIVE_DIR = DRIVE_DATA_ROOT / "tar_datasets"

# GNU tar: 0 = clean, 1 = non-fatal difference/warning, 2 = fatal. Anything >= 2 is a real failure.
TAR_OK_RETURNCODES = (0, 1)

# Flip to True to let tar skip files it genuinely cannot read instead of aborting. Not needed for
# "file changed as we read it" (that is handled above) - this is for permission/IO errors, and it
# silently drops data, so check what got skipped.
TAR_IGNORE_READ_ERRORS = False


def free_gb(path) -> float:
    try:
        return shutil.disk_usage(path).free / 1e9
    except OSError:
        return float("nan")  # Drive FUSE does not always report usage


def run_capture(cmd, shell=False):
    """Run a command capturing output, so a failure is explained in the cell, not the kernel log."""
    return subprocess.run(cmd, shell=shell, capture_output=True, text=True)


def show_stderr(proc, prefix="    ") -> None:
    lines = (proc.stderr or "").strip().splitlines()
    if not lines:
        print(f"{prefix}(no stderr captured)")
        return
    print(f"{prefix}tar wrote {len(lines)} stderr line(s); first 20:")
    for line in lines[:20]:
        print(f"{prefix}  {line}")


def report_failure(proc, what: str) -> None:
    print(f"  FAILED: {what} (exit status {proc.returncode})")
    show_stderr(proc)


def tar_flags() -> list[str]:
    return ["--ignore-failed-read"] if TAR_IGNORE_READ_ERRORS else []


def has_end_of_archive_marker(archive: Path) -> bool:
    """A finished tar ends with at least two 512-byte zero blocks.

    This is the check that actually detects truncation. `tar -t` alone does NOT: given a truncated
    archive it happily lists every entry it can still reach and exits 0, which was verified against
    a deliberately truncated file. Only the end-of-archive marker distinguishes "complete" from
    "cut off partway through".
    """
    if archive.stat().st_size < 1024:
        return False
    with open(archive, "rb") as f:
        f.seek(-1024, os.SEEK_END)
        return f.read(1024) == b"\x00" * 1024


def verify_archive(archive: Path) -> bool:
    """Confirm a freshly built archive is complete and readable before it gets cached on Drive."""
    if not archive.exists():
        print(f"  {archive.name} was never created")
        return False

    if not has_end_of_archive_marker(archive):
        print(f"  {archive.name} has no end-of-archive marker - it is truncated")
        return False

    print(f"  verifying {archive.name} ...")
    start = time.time()
    proc = run_capture(["tar", "-tf", str(archive)])
    if proc.returncode not in TAR_OK_RETURNCODES:
        print(f"  archive failed verification (exit {proc.returncode})")
        show_stderr(proc)
        return False
    entries = len(proc.stdout.splitlines())
    print(f"  verified: {entries:,} entries, end-of-archive marker present ({time.time() - start:.0f}s)")
    return entries > 0


def build_drive_archive(subdir: str, archive: Path) -> bool:
    """Build the reusable archive on Drive. Returns False (without raising) if it fails."""
    partial = archive.with_suffix(".tar.partial")
    partial.unlink(missing_ok=True)  # stale leftover from an earlier failed build

    print(f"Building one-time archive {archive.name} on Drive (slow, but only happens once) ...")
    print(f"  free space - Drive: {free_gb(ARCHIVE_DIR):.1f} GB | local: {free_gb(LOCAL_SCRATCH):.1f} GB")
    cmd = ["tar", "-cf", str(partial), *tar_flags(), "-C", str(DRIVE_DATA_ROOT), subdir]

    start = time.time()
    proc = run_capture(cmd)
    if proc.returncode not in TAR_OK_RETURNCODES:
        report_failure(proc, f"building {archive.name}")
        if partial.exists():
            print(f"    discarding incomplete {partial.name} ({partial.stat().st_size / 1e9:.2f} GB)")
        partial.unlink(missing_ok=True)  # never leave a truncated archive a later run would trust
        return False

    if proc.returncode == 1:
        print(f"  tar exited 1 (non-fatal warnings) after {time.time() - start:.0f}s - "
              f"expected on a Drive mount, archive is still checked below:")
        show_stderr(proc, prefix="  ")

    if not verify_archive(partial):
        print(f"    discarding unverifiable {partial.name}")
        partial.unlink(missing_ok=True)
        return False

    partial.rename(archive)
    print(f"  built in {time.time() - start:.0f}s")
    return True


def extract_from_drive_archive(archive: Path, dst: Path) -> bool:
    size_gb = archive.stat().st_size / 1e9
    local_archive = LOCAL_SCRATCH / archive.name
    print(f"Copying {archive.name} ({size_gb:.2f} GB) from Drive -> local ...")
    start = time.time()
    shutil.copy2(archive, local_archive)
    print(f"  copied in {time.time() - start:.0f}s")

    # A cached archive can predate the truncation check, or have been copied badly - catch that
    # here rather than discovering missing images thousands of batches into an epoch.
    if not has_end_of_archive_marker(local_archive):
        print(f"  {archive.name} is truncated (no end-of-archive marker) - discarding the Drive copy")
        local_archive.unlink(missing_ok=True)
        archive.unlink(missing_ok=True)
        return False

    print(f"Extracting -> {dst} ...")
    start = time.time()
    proc = run_capture(["tar", "-xf", str(local_archive), "-C", str(LOCAL_DATA_ROOT)])
    local_archive.unlink(missing_ok=True)  # reclaim local disk; the Drive copy is the durable one
    if proc.returncode not in TAR_OK_RETURNCODES:
        report_failure(proc, f"extracting {archive.name}")
        return False
    print(f"  extracted in {time.time() - start:.0f}s")
    return True


def stream_drive_to_local(subdir: str) -> bool:
    """Fallback: pipe tar-create straight into tar-extract, writing nothing back to Drive."""
    print(f"Streaming {subdir} directly from Drive -> local (no Drive archive) ...")
    flags = " ".join(tar_flags())
    # `set -o pipefail` would surface a warning-level exit 1 from the producing tar as a pipeline
    # failure, so the producer's status is captured explicitly via PIPESTATUS and judged against
    # TAR_OK_RETURNCODES rather than being treated as fatal.
    pipeline = (
        f"tar -cf - {flags} -C {shlex.quote(str(DRIVE_DATA_ROOT))} {shlex.quote(subdir)} "
        f"| tar -xf - -C {shlex.quote(str(LOCAL_DATA_ROOT))}; "
        f"echo \"__STATUS__ ${{PIPESTATUS[0]}} $?\""
    )
    start = time.time()
    proc = run_capture(["bash", "-c", pipeline])

    status_line = [l for l in proc.stdout.splitlines() if l.startswith("__STATUS__")]
    if not status_line:
        report_failure(proc, f"streaming {subdir} (no status reported)")
        return False
    produce_rc, extract_rc = (int(x) for x in status_line[0].split()[1:3])
    if produce_rc not in TAR_OK_RETURNCODES or extract_rc not in TAR_OK_RETURNCODES:
        print(f"  FAILED: streaming {subdir} (produce exit {produce_rc}, extract exit {extract_rc})")
        show_stderr(proc)
        return False
    if produce_rc == 1 or extract_rc == 1:
        print("  tar exited 1 (non-fatal warnings):")
        show_stderr(proc, prefix="  ")
    print(f"  streamed in {time.time() - start:.0f}s")
    return True


def stage_subdir(subdir: str) -> None:
    src = DRIVE_DATA_ROOT / subdir
    dst = LOCAL_DATA_ROOT / subdir
    marker = dst / ".copy_complete"

    if not src.exists():
        print(f"{src} not found on Drive - skipping")
        return
    if marker.exists():
        print(f"{dst} already staged, skipping")
        return
    if dst.exists():
        shutil.rmtree(dst)  # partial extract from an interrupted run - start over
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    archive = ARCHIVE_DIR / f"{subdir}.tar"

    staged = False
    if archive.exists() or build_drive_archive(subdir, archive):
        staged = extract_from_drive_archive(archive, dst)
    if not staged:
        print(f"  Drive-archive path unavailable for {subdir} - falling back to direct streaming.")
        staged = stream_drive_to_local(subdir)

    if not staged:
        raise RuntimeError(
            f"Could not stage {subdir} to local disk - see the tar errors above. If they are "
            f"'No space left'/quota errors, free space on the Shared Drive (the archives/ folder "
            f"is a good candidate); if they are permission/IO errors on a few files, set "
            f"TAR_IGNORE_READ_ERRORS = True to skip them."
        )

    n_files = sum(1 for p in dst.rglob("*") if p.is_file())
    marker.touch()
    print(f"  {subdir} staged: {n_files:,} files under {dst}\n")


for subdir in ("fv40_merged", "food_ingredients"):
    stage_subdir(subdir)


def localize(path: Path) -> Path:
    return LOCAL_DATA_ROOT / path.relative_to(DRIVE_DATA_ROOT)


train_images = [localize(p) for p in load_image_list(TRAIN_LIST)]
val_images = [localize(p) for p in load_image_list(VAL_LIST)]
test_images = [localize(p) for p in load_image_list(TEST_LIST)]

print(f"train: {len(train_images):,} images | val: {len(val_images):,} | test: {len(test_images):,}")

# Fail loudly here rather than 5,000 batches into an epoch: confirm the staged files actually
# match what the split lists expect.
sampled = train_images[:200] + val_images[:50] + test_images[:50]
missing = [p for p in sampled if not p.exists()]
if missing:
    print(f"\nWARNING - {len(missing)}/{len(sampled)} sampled paths are missing locally, e.g.:")
    for p in missing[:5]:
        print(f"  {p}")
    print("Staging may be incomplete - check the tar output above.")
else:
    print(f"All {len(sampled)} sampled split paths resolve locally.")

# Sanity check the label-path convention against the first training entry
sample_img = train_images[0]
sample_label = image_to_label_path(sample_img)
print(f"\nSample image: {sample_img}")
print(f"Resolved label path: {sample_label}")
print(f"Label exists: {sample_label.exists()}")

## Experiment Configuration

One config block for both arms. Everything a backbone could otherwise differ on is pinned here and
then read by both training runs, so there is a single place to check what was held constant.

The two knobs worth understanding:

- **`TRAIN_SUBSET_FRAC = 0.30`.** Two trainings, one session. This matches runs 1–2's data budget,
  so if you want to sanity-check the ResNet-50 arm against a known number, its closest comparison
  is the baseline notebook (mAP@[0.5:0.95] 0.0785), not run 3's full-data result.
- **`NUM_EPOCHS = 6` with patience 3 on val mAP.** Run 2's post-mortem showed early stopping on
  *val loss* stopped a run before its schedule had done anything, because val loss is computed on
  the natural class distribution while training runs on an oversampled one. Selection and stopping
  both watch val mAP here for the same reason.

In [ ]:
# ---- shared across both backbones (this is the control) -----------------------------------
TRAIN_SUBSET_FRAC = 0.30
RARE_STRATUM_KEEP_ALL = 40   # strata with at most this many images are kept in full

TRAIN_SCALES = (448, 512, 576)
EVAL_SIZE = 576
MAX_SIZE = 800

# torchvision's GeneralizedRCNNTransform samples a random entry from min_size per image while
# training and uses min_size[-1] at eval, so the eval resolution is decided by tuple *order*.
# Force EVAL_SIZE last rather than trusting it - and note this tuple is passed to BOTH
# constructors, overriding the different defaults they ship with (see the model section).
assert EVAL_SIZE in TRAIN_SCALES, "EVAL_SIZE should be one of the training scales"
MIN_SIZE = tuple(s for s in TRAIN_SCALES if s != EVAL_SIZE) + (EVAL_SIZE,)

BATCH_SIZE = 4
NUM_WORKERS = 2

# Linear scaling rule against torchvision's reference recipe (lr=0.02 at batch size 16). Shared
# between arms deliberately; see the caveat in the header.
LR = 0.02 * BATCH_SIZE / 16

NUM_EPOCHS = 6
EARLY_STOP_PATIENCE = 3      # on val mAP, not val loss
OVERSAMPLE_POWER = 0.35      # 0.0 = uniform; 0.5 over-corrected in run 2 and cost head-class AP

CHECKPOINT_EVERY_N_BATCHES = 400
PRINT_EVERY_N_BATCHES = 100
VAL_PRINT_EVERY_N_BATCHES = 100

# ---- the variable under test ---------------------------------------------------------------
# Order matters only for readability of the logs; both arms are independent.
BACKBONES = {
    "resnet50": {
        "label": "ResNet-50 FPN v2",
        "builder": fasterrcnn_resnet50_fpn_v2,
        "weights": FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
        "coco_map": 46.7,   # torchvision's reported COCO box mAP, for context only
    },
    "mobilenet": {
        "label": "MobileNetV3-Large FPN",
        "builder": fasterrcnn_mobilenet_v3_large_fpn,
        "weights": FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT,
        "coco_map": 32.8,
    },
}

# Checkpoints and metrics are namespaced per backbone, so this notebook cannot overwrite any
# earlier run's artifacts and neither arm can overwrite the other's.
RUN_TAG_PREFIX = "fasterrcnn_backbone"

print("backbones        : " + ", ".join(f"{k} ({v['label']})" for k, v in BACKBONES.items()))
print(f"train subset     : {TRAIN_SUBSET_FRAC:.0%}")
print(f"train scales     : {MIN_SIZE} (random per image), max_size={MAX_SIZE}")
print(f"eval size        : {EVAL_SIZE} (= min_size[-1]; used for val mAP, test mAP and latency)")
print(f"batch size / LR  : {BATCH_SIZE} / {LR}")
print(f"epochs           : up to {NUM_EPOCHS}, early stop patience {EARLY_STOP_PATIENCE} on val mAP")
print(f"oversample power : {OVERSAMPLE_POWER}")

## Training Subset (Compute Budget)

Stratified by each image's **rarest** present class, so thinning the split cannot quietly delete a
tail class: strata with `<= RARE_STRATUM_KEEP_ALL` images are kept whole, everything else is
sampled at `TRAIN_SUBSET_FRAC`. `class_counts` computed here (frequency over the *full* training
split) is reused later to split classes into head and tail for the per-class comparison.

In [ ]:
from collections import Counter, defaultdict


def read_class_ids(label_path: Path) -> set[int]:
    """Class ids present in a YOLO label file; empty set for missing/empty (background) files."""
    if not label_path.exists():
        return set()
    ids = set()
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) >= 5:
            ids.add(int(parts[0]))
    return ids


train_images_full = train_images  # keep the unsubsetted list around for reference/reporting

print(f"Scanning {len(train_images_full):,} training label files ...")
start = time.time()
image_class_ids = [read_class_ids(image_to_label_path(p)) for p in train_images_full]
ids_by_path = dict(zip(train_images_full, image_class_ids))
print(f"  done in {time.time() - start:.0f}s")

# Global class frequency (images containing the class) drives both stratum assignment and the
# sampler weights below.
class_counts = Counter(cid for ids in image_class_ids for cid in ids)

# Anchoring on the rarest present class means an image is only ever counted toward the class most
# at risk of disappearing from the subset - and, later, gets the sampling weight of that class.
strata = defaultdict(list)
stratum_by_path = {}
for path, ids in zip(train_images_full, image_class_ids):
    stratum = min(ids, key=lambda c: (class_counts[c], c)) if ids else "__background__"
    stratum_by_path[path] = stratum
    strata[stratum].append(path)

if TRAIN_SUBSET_FRAC >= 1.0:
    print(f"\nTRAIN_SUBSET_FRAC={TRAIN_SUBSET_FRAC} - using all {len(train_images_full):,} training images")
else:
    rng = random.Random(SEED)  # own RNG so the subset is stable regardless of other random() calls
    subset = []
    kept_whole = 0
    for stratum in sorted(strata, key=str):
        members = sorted(strata[stratum])  # sort first so sampling doesn't depend on file order
        if len(members) <= RARE_STRATUM_KEEP_ALL:
            subset.extend(members)
            kept_whole += 1
        else:
            k = max(1, round(TRAIN_SUBSET_FRAC * len(members)))
            subset.extend(rng.sample(members, k))

    rng.shuffle(subset)
    train_images = subset

    # Class coverage in the chosen subset (cheap - labels are already parsed).
    subset_class_counts = Counter(cid for p in train_images for cid in ids_by_path[p])

    print(f"\nStratified subset: {len(train_images):,} / {len(train_images_full):,} images "
          f"({len(train_images) / len(train_images_full):.1%})")
    print(f"  {len(strata)} strata, {kept_whole} kept in full (<= {RARE_STRATUM_KEEP_ALL} images)")
    print(f"  class coverage: {len(subset_class_counts)} / {len(class_counts)} classes still present")
    missing = sorted(set(class_counts) - set(subset_class_counts))
    if missing:
        print(f"  WARNING - classes dropped entirely: {[CLASS_NAMES[c] for c in missing]}")

## Dataset

Unchanged from the earlier runs. YOLO `cx cy w h` (normalised) is converted to absolute `xyxy`,
degenerate boxes are dropped before they reach the model, labels are shifted +1 for background,
and training images get a 50% horizontal flip with boxes mirrored to match.

In [ ]:
class YoloDetectionDataset(Dataset):
    def __init__(self, image_paths: list[Path], train: bool = False):
        self.image_paths = image_paths
        self.train = train

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []
        label_path = image_to_label_path(img_path)
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = min(max((xc - bw / 2) * w, 0), w)
                y1 = min(max((yc - bh / 2) * h, 0), h)
                x2 = min(max((xc + bw / 2) * w, 0), w)
                y2 = min(max((yc + bh / 2) * h, 0), h)
                # Clamping to image bounds (or a tiny/degenerate label box) can leave zero
                # width/height, which torchvision's RCNN forward pass rejects outright.
                if x2 - x1 <= 0 or y2 - y1 <= 0:
                    continue
                boxes.append([x1, y1, x2, y2])
                labels.append(cls_id + 1)  # shift so 0 stays reserved for background

        if self.train and boxes and random.random() < 0.5:
            img = TF.hflip(img)
            boxes = [[w - x2, y1, w - x1, y2] for x1, y1, x2, y2 in boxes]

        boxes_t = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels_t = torch.as_tensor(labels, dtype=torch.int64)
        area = (boxes_t[:, 2] - boxes_t[:, 0]) * (boxes_t[:, 3] - boxes_t[:, 1])

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64),
        }

        return TF.to_tensor(img), target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = YoloDetectionDataset(train_images, train=True)
val_dataset = YoloDetectionDataset(val_images, train=False)
test_dataset = YoloDetectionDataset(test_images, train=False)

print(f"train: {len(train_dataset):,} | val: {len(val_dataset):,} | test: {len(test_dataset):,}")

## Dataloaders & Class-Balanced Sampling

Both arms consume the *same* loaders — same sampler, same seeded draw order — so neither backbone
sees a luckier stream of images than the other.

In [ ]:
if OVERSAMPLE_POWER > 0:
    # Weight by the size of the image's stratum *within the training set actually being used*, so
    # a class thinned by subsetting is weighted on what the model will really see, not on the full
    # split's frequency.
    subset_stratum_sizes = Counter(stratum_by_path[p] for p in train_images)
    sample_weights = torch.tensor(
        [(1.0 / subset_stratum_sizes[stratum_by_path[p]]) ** OVERSAMPLE_POWER for p in train_images],
        dtype=torch.double,
    )
    sampler_generator = torch.Generator().manual_seed(SEED)  # reproducible draw order
    train_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_images),  # same epoch length as uniform shuffling
        replacement=True,
        generator=sampler_generator,
    )

    # Show the effect rather than asserting it: expected draws per epoch for the rarest vs the most
    # common stratum, which is the number that decides whether this is doing anything.
    expected = sample_weights / sample_weights.sum() * len(train_images)
    by_size = sorted(subset_stratum_sizes.items(), key=lambda kv: kv[1])

    def stratum_name(s):
        return "background" if s == "__background__" else CLASS_NAMES[s]

    print(f"Class-balanced sampling on (power={OVERSAMPLE_POWER}): "
          f"{len(subset_stratum_sizes)} strata over {len(train_images):,} images")
    print(f"  expected draws/epoch per image: {expected.min():.2f} (largest stratum) "
          f"to {expected.max():.2f} (smallest)")
    print("  rarest strata  : " + ", ".join(f"{stratum_name(s)} (n={n})" for s, n in by_size[:5]))
    print("  largest strata : " + ", ".join(f"{stratum_name(s)} (n={n})" for s, n in by_size[-5:]))
else:
    train_sampler = None
    print("Class-balanced sampling off (OVERSAMPLE_POWER=0) - uniform shuffle, as in the baseline.")

# shuffle and sampler are mutually exclusive in DataLoader; the sampler already randomizes order.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=(train_sampler is None), sampler=train_sampler,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, collate_fn=collate_fn)

## Model Builders

Both detectors are the *same* Faster R-CNN: same RPN, same two-stage RoI head, same
`FastRCNNPredictor` swapped in for the 148-class taxonomy. Only the feature extractor differs.

Three things about this pairing are easy to get wrong, so they are handled explicitly:

1. **The two constructors ship different default resolutions.** `fasterrcnn_mobilenet_v3_large_fpn`
   defaults to `min_size=320` in some torchvision versions and 800 in others, while the ResNet-50
   builder defaults to 800. Left to the defaults, the "backbone comparison" would silently also be
   a resolution comparison — and MobileNet's speed advantage would be inflated by running on ~3x
   fewer pixels. `MIN_SIZE`/`MAX_SIZE` are therefore passed to both, and the resolution each model
   actually holds is asserted below rather than assumed.
2. **The two arms use different box heads** — ResNet-50 FPN v2 ships the v2 recipe's
   `FastRCNNConvFCHead`, MobileNet the older `TwoMLPHead` — so the predictor's input width is read
   off the constructed model rather than hardcoded. (Both happen to expose 1024 features today,
   which is exactly the kind of coincidence that makes a hardcoded constant break silently later.)
3. **`trainable_backbone_layers` defaults differ** (3 of 5 for ResNet-50, 3 of 6 for MobileNet), so
   the two arms fine-tune a different *fraction* of their backbone. That is inherent to the
   architectures rather than a knob this experiment sets; the trainable-parameter counts are
   printed so the difference is visible instead of hidden.

In [ ]:
def build_model(backbone_key: str, num_classes: int = NUM_CLASSES):
    """Faster R-CNN with the requested backbone, its COCO weights, and this run's resolution."""
    spec = BACKBONES[backbone_key]
    model = spec["builder"](
        weights=spec["weights"],
        min_size=MIN_SIZE,   # passed explicitly: the constructors' own defaults disagree
        max_size=MAX_SIZE,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features  # read, never hardcoded
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Confirm the transform really took the requested sizes rather than the constructor's default.
    tf = model.transform
    assert tuple(tf.min_size) == tuple(MIN_SIZE) and tf.max_size == MAX_SIZE, (
        f"{backbone_key}: transform is min_size={tuple(tf.min_size)}, max_size={tf.max_size} - "
        f"expected {tuple(MIN_SIZE)}/{MAX_SIZE}. The arms would not be comparable."
    )
    return model


def param_counts(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone = sum(p.numel() for p in model.backbone.parameters())
    return {"total": total, "trainable": trainable, "backbone": backbone}


def fpn_levels(model) -> float:
    """How many backbone stages feed the FPN - 4 for ResNet-50, 2 for MobileNetV3-Large.

    This is the structural reason the two arms differ on small objects: fewer pyramid levels means
    fewer distinct scales the RPN gets to propose at.
    """
    fpn = getattr(model.backbone, "fpn", None)
    return float(len(fpn.inner_blocks)) if fpn is not None else float("nan")


# Build both once up front: it surfaces a download/API failure now rather than an hour into the
# first arm's training, and the parameter table is itself part of the comparison.
MODEL_STATS = {}
for key, spec in BACKBONES.items():
    m = build_model(key)
    MODEL_STATS[key] = param_counts(m)
    MODEL_STATS[key]["fpn_levels"] = fpn_levels(m)
    del m   # freed here; each arm builds its own model when it trains

if device.type == "cuda":
    torch.cuda.empty_cache()

stats_df = pd.DataFrame(MODEL_STATS).T
stats_df.index = [BACKBONES[k]["label"] for k in stats_df.index]
stats_df["trainable %"] = 100 * stats_df["trainable"] / stats_df["total"]
print(f"Both models built at min_size={MIN_SIZE}, max_size={MAX_SIZE}, {NUM_CLASSES} outputs\n")
print(stats_df.to_string(formatters={
    "total": "{:,.0f}".format, "trainable": "{:,.0f}".format,
    "backbone": "{:,.0f}".format, "fpn_levels": "{:.0f}".format,
    "trainable %": "{:.1f}%".format,
}))
print(f"\nParameter ratio (ResNet-50 / MobileNet): "
      f"{MODEL_STATS['resnet50']['total'] / MODEL_STATS['mobilenet']['total']:.2f}x")

## Training Loop

The same three functions the earlier runs used, unchanged in behaviour and now taking the model as
an argument so both arms share one implementation:

- `train_one_epoch` — AMP forward/backward, periodic mid-epoch checkpoints, and two first-batch
  heartbeats that localise a stall (before the first batch = data loading; between the two = GPU
  compute). Every print flushes, because block-buffered stdout in a long cell is indistinguishable
  from a hang, which cost real debugging time on the earlier runs.
- `evaluate_map` — COCO mAP in `eval()` mode, `class_metrics=False` for speed; this is the
  checkpoint-selection criterion.
- `evaluate_loss` — Faster R-CNN only returns losses in `train()` mode, so this flips modes under
  `no_grad()` purely to keep the loss curves legible. It decides nothing.

In [ ]:
use_amp = device.type == "cuda"   # T4 fp16 tensor cores are ~8x its fp32 throughput


def log(msg):
    print(msg, flush=True)


def train_one_epoch(model, loader, optimizer, device, scaler,
                    checkpoint_fn=None, checkpoint_every=None, print_every=None):
    model.train()
    running_loss = 0.0
    epoch_start = time.time()
    log(f"  epoch start: {len(loader):,} batches x {loader.batch_size} images "
        f"({len(loader.dataset):,} total), num_workers={loader.num_workers}, "
        f"lr={optimizer.param_groups[0]['lr']:.6f}")

    for batch_idx, (images, targets) in enumerate(loader):
        if batch_idx == 0:
            log(f"  [heartbeat] first batch loaded at {time.time() - epoch_start:.1f}s")

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=scaler is not None):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item()

        if batch_idx == 0:
            log(f"  [heartbeat] first optimizer step done at {time.time() - epoch_start:.1f}s "
                f"- loss {loss.item():.4f}")

        if print_every and (batch_idx + 1) % print_every == 0:
            elapsed = time.time() - epoch_start
            imgs_per_sec = (batch_idx + 1) * loader.batch_size / elapsed
            eta_min = (len(loader) - batch_idx - 1) * elapsed / (batch_idx + 1) / 60
            log(f"  batch {batch_idx + 1}/{len(loader)} - avg_loss: {running_loss / (batch_idx + 1):.4f} "
                f"- {imgs_per_sec:.2f} img/s - {elapsed:.0f}s elapsed - ~{eta_min:.0f} min left this epoch")

        if checkpoint_fn is not None and checkpoint_every and (batch_idx + 1) % checkpoint_every == 0:
            checkpoint_fn()

    return running_loss / len(loader)


@torch.no_grad()
def evaluate_map(model, loader, device, print_every=None, class_metrics=False):
    """COCO-style mAP over a split, in eval() mode - the checkpoint-selection criterion.

    class_metrics=False by default: the per-class breakdown is what makes torchmetrics' compute()
    slow, and per-epoch this only has to rank checkpoints. The test-set call at the end passes
    class_metrics=True. Runs in fp32 rather than under autocast, so the epoch-wise number is
    produced the same way the final reported one is.
    """
    was_training = model.training
    model.eval()
    metric = MeanAveragePrecision(box_format="xyxy", class_metrics=class_metrics)
    start = time.time()
    log(f"  scoring mAP on {len(loader.dataset):,} images ({len(loader):,} batches) ...")
    for batch_idx, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        preds = model(images)
        preds = [{k: v.cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{k: v for k, v in t.items() if k in ("boxes", "labels")} for t in targets]
        metric.update(preds, targets_cpu)

        if print_every and (batch_idx + 1) % print_every == 0:
            log(f"    mAP batch {batch_idx + 1}/{len(loader)} - {time.time() - start:.0f}s elapsed")

    out = metric.compute()
    model.train(was_training)
    return out


@torch.no_grad()
def evaluate_loss(model, loader, device, scaler, print_every=None):
    """Comparable validation loss. Faster R-CNN only returns losses in train() mode, so this
    switches modes to get one - no gradients are taken, the whole function is under no_grad()."""
    model.train()
    running_loss = 0.0
    start = time.time()
    log(f"  validating on {len(loader.dataset):,} images ({len(loader):,} batches) ...")
    for batch_idx, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=scaler is not None):
            loss_dict = model(images, targets)
        running_loss += sum(loss_dict.values()).item()

        if print_every and (batch_idx + 1) % print_every == 0:
            log(f"    val batch {batch_idx + 1}/{len(loader)} - {time.time() - start:.0f}s elapsed")

    return running_loss / len(loader)

## Per-Backbone Checkpointing

Two trainings in one session doubles the exposure to a Colab disconnect, so the artifacts are
organised so that no interruption costs more than one epoch of one arm:

- **Checkpoints are written to local disk and synced to Drive at epoch boundaries only.** A ~350 MB
  write to the Drive FUSE mount every few hundred batches stalled training for minutes at a time on
  the earlier runs. `/content` is wiped on a runtime reset, so the Drive copy is what makes cross-
  session resuming possible — it just costs a fraction as many writes.
- **Each arm has its own `RUN_TAG`** (`fasterrcnn_backbone_resnet50` / `..._mobilenet`), so the
  two arms cannot clobber each other and neither can touch the earlier notebooks' checkpoints.
- **Each arm's history and metrics are also written to Drive as JSON** when it finishes. That is
  what makes the comparison cells survive a disconnect: if ResNet-50 trained yesterday and the
  session died during MobileNet, re-running trains only MobileNet and the comparison still has both
  columns. It also means the analysis further down can be re-run in a fresh session with no GPU
  time at all.

In [ ]:
LOCAL_CKPT_ROOT = Path("/content/checkpoints")
DRIVE_CKPT_ROOT = MERGED_YOLO_DIR.parent / "checkpoints"

RESUME_NAME = "fasterrcnn_resume.pt"
BEST_NAME = "fasterrcnn_best.pt"
LAST_NAME = "fasterrcnn_last.pt"
RECORD_NAME = "run_record.json"


def run_tag(backbone_key: str) -> str:
    return f"{RUN_TAG_PREFIX}_{backbone_key}"


def ckpt_dirs(backbone_key: str) -> tuple[Path, Path]:
    local = LOCAL_CKPT_ROOT / run_tag(backbone_key)
    drive_dir = DRIVE_CKPT_ROOT / run_tag(backbone_key)
    local.mkdir(parents=True, exist_ok=True)
    drive_dir.mkdir(parents=True, exist_ok=True)
    return local, drive_dir


def resolve_checkpoint(backbone_key: str, name: str) -> Path | None:
    """Prefer the local copy (same session, always at least as fresh); fall back to Drive."""
    local, drive_dir = ckpt_dirs(backbone_key)
    if (local / name).exists():
        return local / name
    return (drive_dir / name) if (drive_dir / name).exists() else None


def sync_to_drive(backbone_key: str, name: str) -> None:
    local, drive_dir = ckpt_dirs(backbone_key)
    src = local / name
    if not src.exists():
        return
    start = time.time()
    shutil.copy2(src, drive_dir / name)
    log(f"  [synced {name} to Drive in {time.time() - start:.0f}s]")


def save_run_record(backbone_key: str, record: dict) -> None:
    """Persist an arm's history/metrics to Drive so the analysis survives a lost session."""
    local, drive_dir = ckpt_dirs(backbone_key)
    payload = json.dumps(record, indent=2, default=float)
    (local / RECORD_NAME).write_text(payload)
    (drive_dir / RECORD_NAME).write_text(payload)


def load_run_record(backbone_key: str) -> dict | None:
    local, drive_dir = ckpt_dirs(backbone_key)
    for path in (local / RECORD_NAME, drive_dir / RECORD_NAME):
        if path.exists():
            return json.loads(path.read_text())
    return None


# Set True to ignore existing checkpoints and train both arms from COCO-pretrained weights.
#
# This matters more than it looks: `optimizer.load_state_dict` overwrites the LR in param_groups
# with whatever the checkpoint held, so resuming a checkpoint written under different settings
# silently reinstates that run's LR and defeats the `LR` configured above. Start clean whenever any
# training setting changes - including NUM_EPOCHS, which sets the cosine schedule's T_max.
FRESH_START = True

for key in BACKBONES:
    local, drive_dir = ckpt_dirs(key)
    existing = [n for n in (RESUME_NAME, BEST_NAME, RECORD_NAME)
                if resolve_checkpoint(key, n) is not None]
    print(f"{key:>10}: {local}  |  existing artifacts: {existing or 'none'}")
print(f"\nFRESH_START = {FRESH_START}"
      + ("  (any existing checkpoint above will be ignored, not resumed)" if FRESH_START else
         "  (training will resume from the checkpoints above)"))

## Training Both Arms

`train_backbone()` is the whole per-arm run: build the model, fine-tune from COCO weights, select
on val mAP, stop early on patience, and write a JSON record. The two calls below differ **only** in
which entry of `BACKBONES` they are given.

Two properties worth stating, because they are what make the comparison trustworthy:

- **Each arm gets its own freshly seeded RNG state.** Python/NumPy/Torch seeds are re-set to `SEED`
  at the start of every arm, so the second one trained does not inherit a different augmentation or
  sampler stream than the first. Without this, "trained second" would be a confound.
- **Epoch wall-clock is recorded per arm**, so training cost — not just inference latency — is part
  of the result. On this dataset MobileNet is expected to train several times faster, and if it
  lands anywhere near ResNet-50's accuracy, that is a large part of the practical argument.

Re-running this cell is safe: an arm that already has a completed record on Drive is skipped
(unless `FRESH_START` is True), and an arm interrupted mid-training resumes from its own checkpoint
when `FRESH_START` is False.

In [ ]:
def train_backbone(backbone_key: str) -> dict:
    spec = BACKBONES[backbone_key]
    local_dir, drive_dir = ckpt_dirs(backbone_key)
    log("\n" + "=" * 88)
    log(f"TRAINING: {spec['label']}  (tag {run_tag(backbone_key)})")
    log("=" * 88)

    # Re-seed per arm so both see the same augmentation/sampler stream - "trained second" must not
    # become a hidden variable.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    model = build_model(backbone_key).to(device)
    stats = param_counts(model)
    log(f"  {stats['total']:,} parameters ({stats['trainable']:,} trainable)")

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.SGD(params, lr=LR, momentum=0.9, weight_decay=0.0005)
    # eta_min = LR/100 rather than 0: a final epoch at lr=0 contributes nothing. T_max is stepped
    # once per epoch, so the cosine spans exactly NUM_EPOCHS - changing NUM_EPOCHS changes the
    # schedule, which is why FRESH_START exists.
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=LR / 100)
    scaler = torch.amp.GradScaler(device.type) if use_amp else None

    history = []
    best_val_map = -float("inf")
    best_epoch = 0
    best_val_loss = float("inf")
    best_loss_epoch = 0
    epochs_since_improve = 0
    start_epoch = 0
    stopped_early = False

    resume_path = None if FRESH_START else resolve_checkpoint(backbone_key, RESUME_NAME)
    if resume_path is not None:
        log(f"  resuming from {resume_path}")
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        lr_scheduler.load_state_dict(ckpt["lr_scheduler"])
        if use_amp and ckpt.get("scaler") is not None:
            scaler.load_state_dict(ckpt["scaler"])
        start_epoch = ckpt["epoch"]
        history = ckpt["history"]
        # Patience has to survive the resume, or it restarts at zero every session and the run
        # never stops early.
        epochs_since_improve = ckpt.get("epochs_since_improve", 0)
        best_epoch = ckpt.get("best_epoch", 0)
        best_val_map = ckpt.get("best_val_map", -float("inf"))
        best_val_loss = ckpt.get("best_val_loss", float("inf"))
        best_loss_epoch = ckpt.get("best_loss_epoch", 0)
        log(f"  resumed at epoch {start_epoch + 1}; best val mAP {best_val_map:.4f} (epoch {best_epoch}); "
            f"LR from checkpoint {optimizer.param_groups[0]['lr']:.6f} (configured base {LR})")
    else:
        log("  training from COCO-pretrained weights")

    def save_checkpoint(epoch_for_resume, to_drive=False):
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "lr_scheduler": lr_scheduler.state_dict(),
            "scaler": scaler.state_dict() if use_amp else None,
            "epoch": epoch_for_resume,
            "best_val_map": best_val_map,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "best_loss_epoch": best_loss_epoch,
            "epochs_since_improve": epochs_since_improve,
            "history": history,
        }, local_dir / RESUME_NAME)
        log(f"  [checkpoint saved locally - resume epoch {epoch_for_resume}]")
        if to_drive:
            sync_to_drive(backbone_key, RESUME_NAME)

    for epoch in range(start_epoch, NUM_EPOCHS):
        start = time.time()
        log(f"\n[{backbone_key}] Epoch {epoch + 1}/{NUM_EPOCHS}")
        epoch_lr = optimizer.param_groups[0]["lr"]   # read before the scheduler steps
        train_loss = train_one_epoch(
            model, train_loader, optimizer, device, scaler,
            checkpoint_fn=lambda: save_checkpoint(epoch),   # mid-epoch: local only
            checkpoint_every=CHECKPOINT_EVERY_N_BATCHES,
            print_every=PRINT_EVERY_N_BATCHES,
        )
        train_seconds = time.time() - start   # training only, so the cost comparison is like-for-like

        val_loss = evaluate_loss(model, val_loader, device, scaler,
                                 print_every=VAL_PRINT_EVERY_N_BATCHES)
        val_out = evaluate_map(model, val_loader, device, print_every=VAL_PRINT_EVERY_N_BATCHES)
        val_map, val_map_50 = float(val_out["map"]), float(val_out["map_50"])
        lr_scheduler.step()
        elapsed = time.time() - start

        history.append({"epoch": epoch + 1, "lr": epoch_lr, "train_loss": train_loss,
                        "val_loss": val_loss, "val_map": val_map, "val_map_50": val_map_50,
                        "train_seconds": train_seconds, "seconds": elapsed})
        log(f"[{backbone_key}] Epoch {epoch + 1}/{NUM_EPOCHS} done - lr: {epoch_lr:.6f} "
            f"- train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} "
            f"- val_mAP: {val_map:.4f} (@0.5: {val_map_50:.4f}) "
            f"- {train_seconds:.0f}s train / {elapsed:.0f}s total")

        # Diagnostic only - kept because a disagreement between the two criteria is the evidence
        # for having switched selection away from val loss.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_loss_epoch = epoch + 1

        if val_map > best_val_map:
            log(f"  val mAP improved {best_val_map:.4f} -> {val_map:.4f} - saving best checkpoint")
            best_val_map = val_map
            best_epoch = epoch + 1
            epochs_since_improve = 0
            torch.save(model.state_dict(), local_dir / BEST_NAME)
            sync_to_drive(backbone_key, BEST_NAME)
        else:
            epochs_since_improve += 1
            log(f"  no improvement on val mAP {best_val_map:.4f} (epoch {best_epoch}) - "
                f"{epochs_since_improve}/{EARLY_STOP_PATIENCE} toward early stop")

        save_checkpoint(epoch + 1, to_drive=True)   # epoch boundary: survive a runtime reset

        if epochs_since_improve >= EARLY_STOP_PATIENCE:
            stopped_early = True
            log(f"\n[{backbone_key}] Early stop: {epochs_since_improve} epochs without improving on "
                f"val mAP {best_val_map:.4f} (epoch {best_epoch}). Stopped after epoch {epoch + 1}.")
            break

    torch.save(model.state_dict(), local_dir / LAST_NAME)
    sync_to_drive(backbone_key, LAST_NAME)

    trained_epochs = history[-1]["epoch"] if history else 0
    total_hours = sum(h["seconds"] for h in history) / 3600
    train_hours = sum(h["train_seconds"] for h in history) / 3600
    log(f"\n[{backbone_key}] Done: {trained_epochs} epochs in {total_hours:.2f} h "
        f"({train_hours:.2f} h of it training, the rest validation) - "
        f"{'stopped early' if stopped_early else 'reached NUM_EPOCHS'}.")
    log(f"  Selected checkpoint: epoch {best_epoch}, val mAP {best_val_map:.4f}.")

    record = {
        "backbone": backbone_key,
        "label": spec["label"],
        "params": stats,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_map": best_val_map,
        "best_val_loss": best_val_loss,
        "best_loss_epoch": best_loss_epoch,
        "trained_epochs": trained_epochs,
        "stopped_early": stopped_early,
        "total_hours": total_hours,
        "train_hours": train_hours,
        "config": {"train_images": len(train_images), "min_size": list(MIN_SIZE),
                   "max_size": MAX_SIZE, "eval_size": EVAL_SIZE, "batch_size": BATCH_SIZE,
                   "lr": LR, "num_epochs": NUM_EPOCHS, "oversample_power": OVERSAMPLE_POWER},
    }
    save_run_record(backbone_key, record)

    del model, optimizer, scaler   # the next arm needs the VRAM
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return record


RECORDS = {}
for key in BACKBONES:
    existing = load_run_record(key)
    if existing is not None and existing.get("trained_epochs") and not FRESH_START:
        log(f"[{key}] found a completed record ({existing['trained_epochs']} epochs, "
            f"best val mAP {existing['best_val_map']:.4f}) - skipping training. "
            f"Set FRESH_START=True to retrain.")
        RECORDS[key] = existing
        continue
    RECORDS[key] = train_backbone(key)

print("\nTrained arms:", ", ".join(RECORDS))

### Training Curves

Both arms on shared axes, one color per backbone throughout this notebook (blue = ResNet-50,
orange = MobileNet). Line style separates train from val within an arm, so color never has to carry
two meanings at once.

The learning rate is deliberately not plotted: it is identical for both arms by construction (same
cosine schedule, same base LR), so a curve for it would only restate the control.

In [ ]:
BACKBONE_COLORS = {"resnet50": "#2a78d6", "mobilenet": "#eb6834"}   # fixed per entity, never by rank

hist = {k: pd.DataFrame(r["history"]) for k, r in RECORDS.items() if r["history"]}

fig, (ax_loss, ax_map, ax_time) = plt.subplots(1, 3, figsize=(16, 4.6))

for key, df in hist.items():
    c = BACKBONE_COLORS[key]
    label = RECORDS[key]["label"]
    ax_loss.plot(df["epoch"], df["train_loss"], color=c, linewidth=2, marker="o",
                 markersize=4, label=f"{label} - train")
    ax_loss.plot(df["epoch"], df["val_loss"], color=c, linewidth=2, linestyle="--", marker="s",
                 markersize=4, label=f"{label} - val")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss")
ax_loss.set_title("Training and validation loss")
ax_loss.legend(fontsize=8, frameon=False)
ax_loss.grid(alpha=0.25, linewidth=0.6)
ax_loss.set_axisbelow(True)

for key, df in hist.items():
    c = BACKBONE_COLORS[key]
    ax_map.plot(df["epoch"], df["val_map"], color=c, linewidth=2, marker="o", markersize=5,
                label=RECORDS[key]["label"])
    best = RECORDS[key]["best_epoch"]
    if best:
        ax_map.scatter([best], [RECORDS[key]["best_val_map"]], color=c, s=110,
                       facecolors="none", linewidths=2, zorder=3)
        # Direct-label the selected point only - a number on every marker would be noise.
        ax_map.annotate(f"{RECORDS[key]['best_val_map']:.4f} (ep {best})",
                        xy=(best, RECORDS[key]["best_val_map"]), xytext=(6, 6),
                        textcoords="offset points", fontsize=8, color="#3d3d3a")
ax_map.set_xlabel("Epoch")
ax_map.set_ylabel("Val mAP@[0.5:0.95]")
ax_map.set_title("Val mAP (checkpoint-selection criterion)")
ax_map.legend(fontsize=8, frameon=False)
ax_map.grid(alpha=0.25, linewidth=0.6)
ax_map.set_axisbelow(True)

# Cost, in the same frame as accuracy: mean minutes of training per epoch.
keys = list(hist)
minutes = [hist[k]["train_seconds"].mean() / 60 for k in keys]
bars = ax_time.bar([RECORDS[k]["label"].replace(" ", "\n") for k in keys], minutes,
                   color=[BACKBONE_COLORS[k] for k in keys], width=0.55)
for bar, m in zip(bars, minutes):
    ax_time.text(bar.get_x() + bar.get_width() / 2, m, f"{m:.0f} min",
                 ha="center", va="bottom", fontsize=9, color="#3d3d3a")
ax_time.set_ylabel("Minutes per epoch (training only)")
ax_time.set_title("Training cost per epoch")
ax_time.margins(y=0.15)
ax_time.grid(alpha=0.25, axis="y", linewidth=0.6)
ax_time.set_axisbelow(True)

plt.suptitle(f"Backbone comparison - {len(train_images):,} train images, "
             f"eval size {EVAL_SIZE}, identical schedule")
plt.tight_layout()
plt.show()

pd.concat({RECORDS[k]["label"]: hist[k].set_index("epoch") for k in hist}, names=["backbone"])

## Test-Set mAP and Inference Latency

Each arm's selected checkpoint is evaluated on the held-out test split, and timed. The latency
protocol matches the earlier runs exactly, since a speed claim is only meaningful if measured the
same way for both models:

- **Batch size 1 in `eval()` mode** — the deployment-shaped case. Batching favours the heavier model
  and would understate MobileNet's advantage.
- **`torch.cuda.synchronize()` on both sides of the forward pass.** CUDA launches are asynchronous;
  timing without synchronising measures queueing, not compute, and is typically off by an order of
  magnitude.
- **Warm-up images discarded** (cuDNN autotuning, allocator growth).
- **Model-only** — the image is already decoded and on the GPU, so this covers resize, backbone,
  RPN, heads and NMS, and excludes disk read and JPEG decode.
- **The same `EVAL_SIZE` for both.** Latency moves with input size, so an unequal resolution would
  make the comparison meaningless — this is the trap the model section guards against.

This cell can be re-run in a fresh session with no training: it loads each arm's best checkpoint
from Drive.

In [ ]:
LATENCY_WARMUP_N = 10
LATENCY_SAMPLE_N = 100

try:      # so this cell stands alone in a fresh session where nothing was trained
    RECORDS
except NameError:
    RECORDS = {}


@torch.no_grad()
def measure_latency(model) -> pd.Series:
    """Per-image forward latency, batch size 1, warm-up discarded, CUDA-synchronised."""
    model.eval()
    lat_rng = random.Random(SEED)   # same images for every arm
    n_needed = min(LATENCY_WARMUP_N + LATENCY_SAMPLE_N, len(test_dataset))
    indices = lat_rng.sample(range(len(test_dataset)), n_needed)

    latencies_ms = []
    for i, idx in enumerate(indices):
        img, _ = test_dataset[idx]
        img = img.to(device)   # transfer excluded from the timed region - this is model latency
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model([img])
        if device.type == "cuda":
            torch.cuda.synchronize()
        dt_ms = (time.perf_counter() - t0) * 1000
        if i >= LATENCY_WARMUP_N:
            latencies_ms.append(dt_ms)
    return pd.Series(latencies_ms)


for key, spec in BACKBONES.items():
    best_path = resolve_checkpoint(key, BEST_NAME)
    assert best_path is not None, (
        f"No {BEST_NAME} for {key} in {ckpt_dirs(key)[0]} or {ckpt_dirs(key)[1]} - "
        f"train this arm for at least one epoch first."
    )
    print(f"\n=== {spec['label']} ===")
    print(f"Evaluating checkpoint: {best_path}")

    model = build_model(key).to(device)
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    metric = MeanAveragePrecision(box_format="xyxy", class_metrics=True)
    with torch.no_grad():
        for images, targets in test_loader:
            images = [img.to(device) for img in images]
            preds = model(images)
            preds = [{k: v.cpu() for k, v in p.items()} for p in preds]
            targets_cpu = [{k: v for k, v in t.items() if k in ("boxes", "labels")} for t in targets]
            metric.update(preds, targets_cpu)
    results = metric.compute()

    latency = measure_latency(model)

    print(f"mAP@[0.5:0.95]: {results['map']:.4f} | mAP@0.5: {results['map_50']:.4f} | "
          f"mAP@0.75: {results['map_75']:.4f} | mAR@100: {results['mar_100']:.4f}")
    print(f"latency ({len(latency)} images, batch 1, {device.type}, eval size {EVAL_SIZE}): "
          f"mean {latency.mean():.1f} ms | median {latency.median():.1f} ms | "
          f"p90 {latency.quantile(0.90):.1f} ms | {1000 / latency.mean():.1f} img/s")

    record = RECORDS.get(key) or load_run_record(key) or {"backbone": key, "label": spec["label"]}
    record.setdefault("params", MODEL_STATS[key])   # in case only a checkpoint survived
    record.setdefault("history", [])
    record["test"] = {
        "map": float(results["map"]), "map_50": float(results["map_50"]),
        "map_75": float(results["map_75"]), "mar_100": float(results["mar_100"]),
        "map_small": float(results["map_small"]), "map_medium": float(results["map_medium"]),
        "map_large": float(results["map_large"]),
    }
    record["latency_ms"] = {
        "mean": float(latency.mean()), "median": float(latency.median()),
        "p90": float(latency.quantile(0.90)), "max": float(latency.max()),
        "n": int(len(latency)),
    }
    # 1-indexed model labels (0 is background); -1 means torchmetrics could not evaluate the class.
    record["per_class"] = {
        "class_ids": [int(c) for c in results["classes"].tolist()],
        "ap": [float(v) for v in results["map_per_class"].tolist()],
    }
    RECORDS[key] = record
    save_run_record(key, record)   # persisted, so the analysis below needs no GPU to re-run

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

## Head-to-Head

The comparison this notebook exists to make. `map_small`/`medium`/`large` are broken out because
that is where the two backbones are expected to diverge most: MobileNet's FPN is built over fewer
backbone stages, and small objects are what a shallower pyramid gives up first — this dataset is
full of small ingredients, so it is a fair place to look.

In [ ]:
ORDER = [k for k in BACKBONES if k in RECORDS and "test" in RECORDS[k]]
assert ORDER, "No arm has test metrics yet - run the evaluation cell above."

rows = {}
for key in ORDER:
    r = RECORDS[key]
    rows[r["label"]] = {
        "mAP@[0.5:0.95]": r["test"]["map"],
        "mAP@0.5": r["test"]["map_50"],
        "mAP@0.75": r["test"]["map_75"],
        "mAR@100": r["test"]["mar_100"],
        "mAP small": r["test"]["map_small"],
        "mAP medium": r["test"]["map_medium"],
        "mAP large": r["test"]["map_large"],
        "latency (ms)": r["latency_ms"]["mean"],
        "throughput (img/s)": 1000 / r["latency_ms"]["mean"],
        "params (M)": r["params"]["total"] / 1e6,
        "train min/epoch": np.mean([h["train_seconds"] for h in r["history"]]) / 60 if r.get("history") else np.nan,
        "best epoch": r.get("best_epoch", np.nan),
        "epochs run": r.get("trained_epochs", np.nan),
    }

comparison = pd.DataFrame(rows)

delta_cols = []
if len(ORDER) == 2:
    a, b = (RECORDS[k]["label"] for k in ORDER)   # column order follows BACKBONES
    delta_col, ratio_col = f"delta ({b} - {a})", f"ratio ({b} / {a})"
    comparison[delta_col] = comparison[b] - comparison[a]
    comparison[ratio_col] = comparison[b] / comparison[a]
    delta_cols = [delta_col, ratio_col]

print(f"Both arms: {len(train_images):,} train images, scales {MIN_SIZE} (eval {EVAL_SIZE}), "
      f"batch {BATCH_SIZE}, LR {LR}, cosine schedule, oversample {OVERSAMPLE_POWER}, "
      f"early stop patience {EARLY_STOP_PATIENCE} on val mAP.\n")

# Counts are rows, not columns, so they get their own row-wise format rather than 4 decimals.
count_rows = [r for r in ("best epoch", "epochs run") if r in comparison.index]
styled = comparison.style.format("{:.4f}")
if delta_cols:
    styled = styled.format("{:+.4f}", subset=pd.IndexSlice[:, [delta_cols[0]]]) \
                   .format("{:.2f}x", subset=pd.IndexSlice[:, [delta_cols[1]]])
styled = styled.format("{:.0f}", subset=pd.IndexSlice[count_rows, :])   # last, so it wins
styled

In [ ]:
fig, (ax_bar, ax_scatter) = plt.subplots(1, 2, figsize=(14.5, 5))

# --- accuracy, metric by metric -------------------------------------------------------------
metrics = ["mAP@[0.5:0.95]", "mAP@0.5", "mAP@0.75", "mAR@100"]
x = np.arange(len(metrics))
width = 0.36
for i, key in enumerate(ORDER):
    vals = [comparison.loc[m, RECORDS[key]["label"]] for m in metrics]
    offset = (i - (len(ORDER) - 1) / 2) * (width + 0.02)   # 2px-equivalent gap between adjacent bars
    bars = ax_bar.bar(x + offset, vals, width, color=BACKBONE_COLORS[key],
                      label=RECORDS[key]["label"])
    for bar, v in zip(bars, vals):
        ax_bar.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom",
                    fontsize=8, color="#3d3d3a")
ax_bar.set_xticks(x, metrics)
ax_bar.set_ylabel("Test-set score")
ax_bar.set_title("Detection quality on the held-out test split")
ax_bar.legend(fontsize=9, frameon=False)
ax_bar.margins(y=0.18)
ax_bar.grid(alpha=0.25, axis="y", linewidth=0.6)
ax_bar.set_axisbelow(True)

# --- the actual trade-off --------------------------------------------------------------------
for key in ORDER:
    r = RECORDS[key]
    ax_scatter.scatter(r["latency_ms"]["mean"], r["test"]["map"], s=160,
                       color=BACKBONE_COLORS[key], zorder=3,
                       edgecolors="white", linewidths=2)   # surface ring keeps overlaps legible
    ax_scatter.annotate(
        f"{r['label']}\n{r['params']['total'] / 1e6:.1f}M params",
        xy=(r["latency_ms"]["mean"], r["test"]["map"]), xytext=(10, -4),
        textcoords="offset points", fontsize=9, color="#3d3d3a")
ax_scatter.set_xlabel(f"Mean inference latency at batch 1 (ms, {device.type}, eval size {EVAL_SIZE})")
ax_scatter.set_ylabel("Test mAP@[0.5:0.95]")
ax_scatter.set_title("Accuracy vs. cost (up and to the left is better)")
ax_scatter.margins(x=0.30, y=0.30)
ax_scatter.grid(alpha=0.25, linewidth=0.6)
ax_scatter.set_axisbelow(True)

plt.tight_layout()
plt.show()

### Where the Gap Lives: Per-Class AP

A single mAP hides which classes each backbone actually gave up. This joins both arms' per-class AP
and splits classes at the median training frequency, the same head/tail convention the earlier
notebooks use.

`torchmetrics` reports `-1` for a class it could not evaluate (no ground-truth boxes *and* no
predictions anywhere in the test split). That means "not measured", not "AP of -1", so those rows
are excluded rather than averaged in or sorted to the bottom as though they were the worst scores.

In [ ]:
per_class = None
for key in ORDER:
    pc = RECORDS[key]["per_class"]
    df = pd.DataFrame({"class_id": pc["class_ids"], RECORDS[key]["label"]: pc["ap"]})
    per_class = df if per_class is None else per_class.merge(df, on="class_id", how="outer")

per_class["class_id"] = per_class["class_id"].astype(int)   # an outer join can float-ify it
per_class["class"] = [CLASS_NAMES[i - 1] for i in per_class["class_id"]]   # ids are 1-indexed
per_class["train_images"] = [class_counts.get(i - 1, 0) for i in per_class["class_id"]]
label_cols = [RECORDS[k]["label"] for k in ORDER]

# Drop classes neither arm could evaluate; keep a class if at least one arm measured it.
measured = per_class[(per_class[label_cols] >= 0).any(axis=1)].copy()
measured[label_cols] = measured[label_cols].mask(measured[label_cols] < 0)   # -1 -> NaN, not 0

tail_threshold = measured["train_images"].median()
measured["group"] = np.where(measured["train_images"] < tail_threshold, "tail", "head")

print(f"{len(CLASS_NAMES)} classes in the taxonomy; {len(measured)} evaluated by at least one arm")
print(f"Head/tail split at the median training frequency ({tail_threshold:.0f} images)\n")

summary = []
for group in ("head", "tail"):
    g = measured[measured["group"] == group]
    row = {"group": group, "classes": len(g)}
    for col in label_cols:
        row[f"{col} mean AP"] = g[col].mean()
        row[f"{col} AP=0"] = int((g[col] == 0).sum())
    summary.append(row)
summary_df = pd.DataFrame(summary).set_index("group")
print(summary_df.to_string(float_format=lambda v: f"{v:.3f}"))

if len(label_cols) == 2:
    a_col, b_col = label_cols
    measured["delta"] = measured[b_col] - measured[a_col]
    both = measured.dropna(subset=[a_col, b_col])
    print(f"\nClasses where {b_col} beats {a_col}: {(both['delta'] > 0).sum()} / {len(both)}")
    print(f"  mean per-class AP: {a_col} {both[a_col].mean():.3f} | {b_col} {both[b_col].mean():.3f}")
    print(f"\nLargest gains for {b_col}:")
    print(both.nlargest(5, "delta")[["class", "train_images", a_col, b_col, "delta"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print(f"\nLargest losses for {b_col}:")
    print(both.nsmallest(5, "delta")[["class", "train_images", a_col, b_col, "delta"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

table = measured.sort_values(label_cols[0], ascending=False)[
    ["class", "train_images", "group"] + label_cols + (["delta"] if len(label_cols) == 2 else [])
]
table.style.hide(axis="index").background_gradient(subset=label_cols, cmap="Blues").format(
    {c: "{:.3f}" for c in label_cols + (["delta"] if len(label_cols) == 2 else [])})

## Sample Predictions, Side by Side

The same test images through both selected checkpoints. Green = ground truth, red = prediction at
score >= 0.5. This is where a mAP gap becomes concrete: typically the same objects found by both,
with the lighter backbone missing the smaller or more crowded instances.

In [ ]:
def draw_predictions(ax, image_tensor, target, prediction=None, score_thresh=0.5):
    ax.imshow(image_tensor.permute(1, 2, 0).numpy())

    for box, label in zip(target["boxes"], target["labels"]):
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2,
                                       edgecolor="lime", facecolor="none"))
        ax.text(x1, max(y1 - 5, 0), f"GT: {CLASS_NAMES[int(label) - 1]}", color="lime", fontsize=8)

    if prediction is not None:
        keep = prediction["scores"] >= score_thresh
        for box, label, score in zip(prediction["boxes"][keep], prediction["labels"][keep],
                                     prediction["scores"][keep]):
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2,
                                           edgecolor="red", facecolor="none"))
            ax.text(x1, y2 + 12, f"{CLASS_NAMES[int(label) - 1]}: {score:.2f}", color="red", fontsize=8)

    ax.axis("off")


sample_rng = random.Random(SEED)
sample_indices = sample_rng.sample(range(len(test_dataset)), min(4, len(test_dataset)))

# One column per backbone, one row per image, so the same scene is compared across arms. figsize
# and dpi are kept modest deliberately: a (16, 10) figure embedded ~2.9 MB of PNG into the .ipynb
# on an earlier run, which dominated the file size in git.
fig, axes = plt.subplots(len(sample_indices), len(ORDER),
                         figsize=(5 * len(ORDER), 4.4 * len(sample_indices)), dpi=90)
axes = np.array(axes).reshape(len(sample_indices), len(ORDER))   # correct for a 1-arm run too

for col, key in enumerate(ORDER):
    model = build_model(key).to(device)
    model.load_state_dict(torch.load(resolve_checkpoint(key, BEST_NAME), map_location=device))
    model.eval()
    with torch.no_grad():
        for row, idx in enumerate(sample_indices):
            img, target = test_dataset[idx]
            pred = {k: v.cpu() for k, v in model([img.to(device)])[0].items()}
            draw_predictions(axes[row, col], img, target, pred)
            if row == 0:
                axes[row, col].set_title(RECORDS[key]["label"], fontsize=11,
                                         color=BACKBONE_COLORS[key])
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

plt.suptitle("Same test images, both backbones - green = ground truth, red = prediction (score >= 0.5)")
plt.tight_layout()
plt.show()

## How to Read This Experiment

**What it establishes.** Every difference between the two result columns is attributable to the
backbone: same images, same seeds, same resolution, same schedule, same sampler, same metric, same
latency protocol. The parameter-count, epoch-time and latency columns make the cost side of the
trade-off explicit, so the conclusion is a point on an accuracy/cost curve rather than a winner.

**What it does not establish.**

- **Neither arm's LR was tuned for it.** A shared LR is the right control for "which backbone",
  but it means neither number is that backbone's *best achievable* result. A MobileNet arm that
  loses by a small margin here might close some of it under its own LR search.
- **These are 30%-subset numbers**, chosen so two trainings fit one session. They are comparable to
  each other and roughly to runs 1–2, but not to run 3's full-data results. Both arms sat on the
  same data wall the earlier runs hit (~51 images per class), which compresses the gap between any
  two architectures — a stronger backbone cannot exploit data that is not there.
- **Latency is single-stream on one GPU class.** MobileNet's advantage is generally larger on CPU
  and on mobile/edge hardware, which is the setting it was designed for and is not measured here.

**How to extend it.** The `BACKBONES` dict is the only thing a third arm needs: add an entry
(`fasterrcnn_mobilenet_v3_large_320_fpn` for the low-resolution variant, or
`fasterrcnn_resnet50_fpn` for the v1 recipe) and re-run. Every downstream cell — curves, table,
plots, per-class join, sample grid — iterates over that dict and picks the new arm up, with the one
caveat that the two-column delta/ratio columns only appear when exactly two arms are present, and
that the fixed color map in `BACKBONE_COLORS` needs a hue for the new key (take the next slot in
the validated categorical order rather than inventing one).